# Silent Signal — chuẩn bị graph cho ASL Citizen top-200

Notebook này **không tải hoặc extract video lại**. Nó đọc 6.146 cache pose `.npz` từ notebook 04, chuyển 133 keypoint thành graph 75 nút, tạo tensor `[64, 75, 7]` và lưu lên Drive. Có thể dừng/chạy lại với `OVERWRITE=False`; bước này dùng CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-graph-preprocessing'  # @param {type:'string'}
RESULTS_ROOT_STR = '/content/drive/MyDrive/silent-signal-results/asl_citizen'  # @param {type:'string'}
RUN_PREPARE = True  # @param {type:'boolean'}
OVERWRITE = False  # @param {type:'boolean'}
LIMIT = 0  # @param {type:'integer'}
EXPECTED_CLIPS = 6146
EXPECTED_MANIFEST_SHA256 = '7e3dc11bc6fb25ff6b4311c9fe4f2175841ce6ad5eab89da4e5c8af0de596dfe'
RESULTS_ROOT = Path(RESULTS_ROOT_STR)
SUBSET_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top200'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
RAW_POSE_ROOT = SUBSET_ROOT / 'pose/rtmpose_l_coco_wholebody_384x288/raw'
GRAPH_ROOT = SUBSET_ROOT / 'graph/asl_citizen_coco_wholebody_v1/t64'
GRAPH_REPORT = SUBSET_ROOT / 'reports/graph_preparation_t64.json'
PROJECT_ROOT = Path('/content/silent-signal')
ENV_ROOT = Path('/content/graph-env')
GRAPH_PY = ENV_ROOT / 'bin/python'

## Cài môi trường CPU độc lập
Dùng Python 3.11 để tránh lỗi binary incompatibility của NumPy trên runtime Colab.

In [ ]:
import shutil, subprocess, sys, time
def run(command):
    command = list(map(str, command))
    started = time.perf_counter()
    print(time.strftime('[%H:%M:%S] START'), ' '.join(command), flush=True)
    subprocess.run(command, check=True)
    elapsed = time.perf_counter() - started
    print(time.strftime('[%H:%M:%S] DONE '), f'{elapsed:.1f}s', flush=True)
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv')
if not GRAPH_PY.exists():
    run([UV, 'venv', '--python', '3.11', ENV_ROOT])
if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch', PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only', 'origin', PROJECT_GIT_REF])
run([UV, 'pip', 'install', '--python', GRAPH_PY, '-e', PROJECT_ROOT])
run([GRAPH_PY, '-c', 'import numpy, silent_signal; print(numpy.__version__)'])

## Kiểm tra đúng Drive và đúng 6.146 cache
Nếu dùng shortcut, sửa `RESULTS_ROOT_STR` thành đường dẫn `/content/drive/.shortcut-targets-by-id/.../silent-signal-results/asl_citizen`.

In [ ]:
import hashlib
if not MANIFEST.is_file(): raise FileNotFoundError(MANIFEST)
if not RAW_POSE_ROOT.is_dir(): raise FileNotFoundError(RAW_POSE_ROOT)
manifest_sha = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
raw_count = 0
scan_started = time.perf_counter()
for raw_count, _ in enumerate(RAW_POSE_ROOT.glob('*/*.npz'), start=1):
    if raw_count % 500 == 0: print(f'[scan pose] {raw_count}/{EXPECTED_CLIPS}', flush=True)
print(f'[scan pose] xong trong {time.perf_counter() - scan_started:.1f}s', flush=True)
print('Manifest SHA-256:', manifest_sha)
print('Raw pose cache:', raw_count, '/', EXPECTED_CLIPS)
if manifest_sha != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Manifest không đúng bản đã dùng để extract pose.')
if raw_count != EXPECTED_CLIPS:
    raise RuntimeError('Pose cache chưa đủ hoặc đang mount nhầm Drive.')

## Tạo graph cache có resume và log `đã xử lý / tổng`

In [ ]:
command = [GRAPH_PY, '-u', '-m', 'silent_signal.cli.prepare_graph',
    '--config', PROJECT_ROOT / 'configs/preprocessing/asl_citizen_graph.yaml',
    '--manifest', MANIFEST, '--pose-root', RAW_POSE_ROOT,
    '--output-root', GRAPH_ROOT, '--report', GRAPH_REPORT,
    '--continue-on-error', '--progress-every', '25']
if LIMIT > 0: command += ['--limit', str(LIMIT)]
if OVERWRITE: command.append('--overwrite')
if RUN_PREPARE: run(command)
else: print('RUN_PREPARE=False; chưa xử lý graph.')

## Kết quả bền vững trên Drive
PASS khi `prepared + resumed = 6146` và `failed = 0`.

In [ ]:
import json
if GRAPH_REPORT.is_file():
    report = json.loads(GRAPH_REPORT.read_text(encoding='utf-8'))
    graph_count = 0
    for graph_count, _ in enumerate(GRAPH_ROOT.glob('*/*.npz'), start=1):
        if graph_count % 500 == 0: print(f'[audit graph] {graph_count}/{EXPECTED_CLIPS}', flush=True)
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('Graph cache trên Drive:', graph_count, '/', EXPECTED_CLIPS)
    if report['failed'] == 0 and report['prepared'] + report['resumed'] == EXPECTED_CLIPS:
        print('PASS: graph preprocessing hoàn tất.')
else:
    print('Chưa có báo cáo:', GRAPH_REPORT)